# MFU / time calculator

Architecture assumptions (matching `dense_mha.yaml`):
- `n_layers = k`, `dmodel = 64 * k`, `dff = 2.5 * dmodel`
- dense FF (`n_experts = 1`, `top_k = 1`), MHA (`kv_heads = n_layers`)
- vocab = 50,304, tied? No — separate input/output embeddings
- Hardware: 4x H100 BF16 @ 1979/2 TFLOPs each (configurable)

FLOPs accounting:
- Param FLOPs: `6 * active_params * tokens`
- Attention FLOPs (not in params): `12 * n_layers * seq_len * dmodel * tokens`

In [15]:
VOCAB = 50_304
H100_BF16 = (1_979 / 2) * 1e12  # FLOP/s per GPU

FWD_BWD = 6  # 2x forward + 4x backward per matmul-input


def model_flops(k: int, seq_len: int, batch_size: int, n_steps: int, dff_factor: float):
    """Returns (active_params, attn_inner, total_flops) for n_steps of training. Assumes MHA."""
    n_layers = k
    dmodel = 64 * k
    dff = dff_factor * dmodel

    attn_params = 4 * dmodel ** 2  # MHA: Q, K, V, O each dmodel x dmodel
    ff_params = 3 * dmodel * dff  # gated MLP (3 matrices)
    embedding = VOCAB * dmodel
    active_params = n_layers * (attn_params + ff_params) + embedding

    # equivalent "weights touched" by each token in the attention QK^T + AV ops
    attn_inner = 2 * n_layers * seq_len * dmodel

    tokens = seq_len * batch_size * n_steps
    param_flops = FWD_BWD * active_params * tokens
    attn_flops  = FWD_BWD * attn_inner     * tokens
    total_flops = param_flops + attn_flops
    return active_params, attn_inner, total_flops


def _fmt_time(seconds: float) -> str:
    return f"{seconds / 60:.1f}min = {seconds / 3600:.2f}h = {seconds / 86400:.2f}d"


def _print_arch(active: float, attn_inner: float):
    total = active + attn_inner
    print(f"active params: {active / 1e6:.1f}M")
    print(f"attn_inner:    {attn_inner / 1e6:.1f}M")
    print(f"attn_inner / total: {attn_inner / total * 100:.1f}%")


def mfu_from_time(
    k: int,
    seq_len: int,
    batch_size: int,
    n_steps: int,
    time_seconds: float,
    n_gpus: int,
    dff_factor: float,
    peak_flops: float = H100_BF16,
) -> float:
    active, attn_inner, th_flops = model_flops(k, seq_len, batch_size, n_steps, dff_factor)
    realized = time_seconds * n_gpus * peak_flops
    mfu = th_flops / realized
    _print_arch(active, attn_inner)
    print(f"theoretical flops: {th_flops:.3e}")
    print(f"realized capacity: {realized:.3e}")
    print(f"time: {_fmt_time(time_seconds)}")
    print(f"GPU-hours: {time_seconds * n_gpus / 3600:.2f}")
    print(f"MFU: {mfu * 100:.2f}%")
    return mfu


def time_from_mfu(
    k: int,
    seq_len: int,
    batch_size: int,
    n_steps: int,
    mfu: float,
    n_gpus: int,
    dff_factor: float,
    peak_flops: float = H100_BF16,
) -> float:
    active, attn_inner, th_flops = model_flops(k, seq_len, batch_size, n_steps, dff_factor)
    seconds = th_flops / (mfu * n_gpus * peak_flops)
    _print_arch(active, attn_inner)
    print(f"theoretical flops: {th_flops:.3e}")
    print(f"time @ MFU={mfu * 100:.1f}%: {_fmt_time(seconds)}")
    print(f"GPU-hours: {seconds * n_gpus / 3600:.2f}")
    return seconds

## Example 1: time → MFU

In [16]:
mfu = mfu_from_time(k=16, seq_len=2048, batch_size=128, n_steps=46_000, time_seconds=15 * 60 ** 2, n_gpus=4, dff_factor=2.5)

active params: 244.4M
attn_inner:    67.1M
attn_inner / total: 21.5%
theoretical flops: 2.254e+19
realized capacity: 2.137e+20
time: 900.0min = 15.00h = 0.62d
GPU-hours: 60.00
MFU: 10.55%


## Example 2: MFU → time

In [17]:
seconds = time_from_mfu(k=16, seq_len=2048, batch_size=128, n_steps=160_001, mfu=0.10, n_gpus=4, dff_factor=2.5)

active params: 244.4M
attn_inner:    67.1M
attn_inner / total: 21.5%
theoretical flops: 7.841e+19
time @ MFU=10.0%: 3301.6min = 55.03h = 2.29d
GPU-hours: 220.11
